In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/sample_submission.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/train.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_49.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_90.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_77.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_66.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_54.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_42.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_81.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_72.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final

In [2]:
import pandas as pd

BASE = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"

train = pd.read_csv(BASE + "/train.csv")
test = pd.read_csv(BASE + "/test.csv")

print("Train data:")
print(train.head())

print("Test data:")
print(test.head())

Train data:
        filename  label
0  audio_192.wav    3.0
1  audio_436.wav    3.0
2  audio_447.wav    3.5
3  audio_126.wav    2.0
4   audio_61.wav    2.0
Test data:
        filename  label
0  audio_128.wav     -1
1  audio_156.wav     -1
2   audio_19.wav     -1
3  audio_108.wav     -1
4  audio_206.wav     -1


In [3]:
print("Train rows:", len(train))
print("Test rows:", len(test))
print("Columns:", train.columns.tolist())

Train rows: 769
Test rows: 216
Columns: ['filename', 'label']


In [4]:
import os

train_audio = BASE + "/train"

files = os.listdir(train_audio)

print("Total audio files:",len (files))
print("First 10 files:")
print(files[:10])

Total audio files: 769
First 10 files:
['audio_49.wav', 'audio_663.wav', 'audio_5059.wav', 'audio_90.wav', 'audio_581.wav', 'audio_698.wav', 'audio_77.wav', 'audio_66.wav', 'audio_555.wav', 'audio_54.wav']


In [5]:
import librosa

audio_path = train_audio + "/" + train.iloc[0]["filename"]

y, sr = librosa.load(audio_path, sr=16000)

print("Audio loaded successfully!")
print("Sample rate:", sr)
print("Audio length:", len(y))

Audio loaded successfully!
Sample rate: 16000
Audio length: 968134


In [6]:
print("Duration:", len(y) / sr,"seconds")
print("Minimum value:", y.min())
print("Maximum value:", y.max())

Duration: 60.508375 seconds
Minimum value: -0.4777832
Maximum value: 0.38607788


In [7]:
import numpy as np

mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

print("MFCC shape:", mfcc.shape)

MFCC shape: (13, 1891)


In [8]:
mfcc_mean = np.mean(mfcc, axis=1)

print("MFCC mean shape:", mfcc_mean.shape)
print(mfcc_mean)

MFCC mean shape: (13,)
[-403.8359     103.57634    -14.213533    16.673616     7.04855
   -4.9996395    5.885733    -9.268934    -8.681059    -6.1113496
  -11.947043    -8.226156    -6.238205 ]


In [9]:
for i, value in enumerate(mfcc_mean):
    print("Feature", i + 1, ":", value)

Feature 1 : -403.8359
Feature 2 : 103.57634
Feature 3 : -14.213533
Feature 4 : 16.673616
Feature 5 : 7.04855
Feature 6 : -4.9996395
Feature 7 : 5.885733
Feature 8 : -9.268934
Feature 9 : -8.681059
Feature 10 : -6.1113496
Feature 11 : -11.947043
Feature 12 : -8.226156
Feature 13 : -6.238205


In [10]:
print(mfcc_mean)
print("Total features:", len(mfcc_mean))

[-403.8359     103.57634    -14.213533    16.673616     7.04855
   -4.9996395    5.885733    -9.268934    -8.681059    -6.1113496
  -11.947043    -8.226156    -6.238205 ]
Total features: 13


In [11]:
print("Print is working!")

Print is working!


In [12]:
features = []

for filename in train["filename"]:
    path = train_audio + "/" + filename
    audio, rate = librosa.load(path, sr=16000)
    mfcc = librosa.feature.mfcc(y=audio, sr=rate, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    features.append(mfcc_mean)

X = np.array(features)
print("Feature matrix shape:", X.shape)

Feature matrix shape: (769, 13)


In [13]:
y_target = train["label"].values
print("Labels shape:", y_target.shape)

Labels shape: (769,)


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y_target, test_size=0.2, random_state=42)

print("Training data:", X_train.shape)
print("Validation data:", X_val.shape)

Training data: (615, 13)
Validation data: (154, 13)


In [15]:
!pip install scikit_learn -q

In [16]:
import sklearn
print(sklearn.__version__)

1.6.1


In [17]:
from sklearn.ensemble import RandomForestRegressor 

model = RandomForestRegressor(n_estimators=100, random_state=42)

model.fit(X_train, y_train)
print("Model training complete")

Model training complete


In [18]:
y_pred = model.predict(X_val)

print("Predictions:", y_pred[:10])

Predictions: [0.    3.85  3.33  0.    3.6   2.88  3.415 4.065 3.78  3.24 ]


In [19]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_val, y_pred)

print("MSE:", mse)

MSE: 0.6829227272727272


In [20]:
from sklearn.metrics import r2_score
r2 = r2_score(y_val, y_pred)
print("R2 score:", r2)

R2 score: 0.627667249435335


In [21]:
test_audio = BASE + "/test"
test_features = []

for filename in test["filename"]:
    path = test_audio + "/" + filename
    audio, rate = librosa.load(path, sr=16000)
    mfcc = librosa.feature.mfcc(y=audio, sr=rate, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    test_features.append(mfcc_mean)
    
X_test = np.array(test_features)
print("Test feature matrix:", X_test.shape)

Test feature matrix: (216, 13)


In [22]:
test_predictions = model.predict(X_test)

print("Total predictions:", len(test_predictions))
print(test_predictions[:10])

Total predictions: 216
[3.255 3.785 3.875 2.885 3.005 3.345 3.53  3.4   3.55  3.495]


In [23]:
submission = pd.DataFrame({"prediction": test_predictions})

print(submission.head())
print("Submission shape:", submission.shape)

   prediction
0       3.255
1       3.785
2       3.875
3       2.885
4       3.005
Submission shape: (216, 1)


In [24]:
submission.to_csv("submission.csv", index=False)
print("sunbmission.csv created succesfully!")

sunbmission.csv created succesfully!


In [25]:
print(submission.head(10))
print("\nshape:", submission.shape)
print("\ncolumns:", submission.columns.tolist())

   prediction
0       3.255
1       3.785
2       3.875
3       2.885
4       3.005
5       3.345
6       3.530
7       3.400
8       3.550
9       3.495

shape: (216, 1)

columns: ['prediction']


In [26]:
print(submission.head(10))

   prediction
0       3.255
1       3.785
2       3.875
3       2.885
4       3.005
5       3.345
6       3.530
7       3.400
8       3.550
9       3.495


In [27]:
print(submission.columns)

Index(['prediction'], dtype='object')


In [28]:
print(type(X_test))
print(X_test.shape)
print(type(test_predictions))
print(len(test_predictions))

<class 'numpy.ndarray'>
(216, 13)
<class 'numpy.ndarray'>
216


In [29]:
import os

print(os.listdir("/kaggle/input"))

['competitions']


In [30]:
import os

print(os.listdir("/kaggle/input/competitions"))

['shl-hiring-assessment-2026']


In [31]:
import os

print(os.listdir("/kaggle/input/competitions/shl-hiring-assessment-2026"))

['Dataset_Final']


In [32]:
import os

print(os.listdir("/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final"))

['sample_submission.csv', 'train.csv', 'test.csv', 'test', 'train']


In [33]:
import pandas as pd

sample = pd.read_csv("/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/sample_submission.csv")
print(sample.head())
print(sample.columns)
print(sample.shape)

         filename  label
0   audio_804.wav     -1
1  audio_1028.wav     -1
2   audio_865.wav     -1
3   audio_774.wav     -1
4  audio_1138.wav     -1
Index(['filename', 'label'], dtype='object')
(204, 2)


In [34]:
test_df = pd.read_csv("/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test.csv")
print(test_df.head())
print(test_df.columns)
print("Test rows:", len(test_df))

        filename  label
0  audio_128.wav     -1
1  audio_156.wav     -1
2   audio_19.wav     -1
3  audio_108.wav     -1
4  audio_206.wav     -1
Index(['filename', 'label'], dtype='object')
Test rows: 216


In [35]:
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": test_predictions})

print(submission.head(10))
print("submission shape:", submission.shape)

        filename  label
0  audio_128.wav  3.255
1  audio_156.wav  3.785
2   audio_19.wav  3.875
3  audio_108.wav  2.885
4  audio_206.wav  3.005
5  audio_161.wav  3.345
6  audio_215.wav  3.530
7  audio_213.wav  3.400
8   audio_11.wav  3.550
9  audio_155.wav  3.495
submission shape: (216, 2)


In [36]:
submission.to_csv("submission.csv", index=False)
print("Final submission.csv created!")
print(submission.head())

Final submission.csv created!
        filename  label
0  audio_128.wav  3.255
1  audio_156.wav  3.785
2   audio_19.wav  3.875
3  audio_108.wav  2.885
4  audio_206.wav  3.005


In [37]:
print(submission.head())
print(submission.shape)

        filename  label
0  audio_128.wav  3.255
1  audio_156.wav  3.785
2   audio_19.wav  3.875
3  audio_108.wav  2.885
4  audio_206.wav  3.005
(216, 2)
